# Medication Identification — Med-ID Mode

Interactive notebook with **live streaming HTML** that mimics the `/refine_with_kg` route.
Agent reasoning, tool calls, tool results, and recommendations render in real-time
using the same `chat_med_refinement.html` UI.

## 0. Setup

In [ ]:
import sys, os, re, json, html as html_mod, uuid
from collections import defaultdict
from typing import List

NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__"))
if NOTEBOOK_DIR not in sys.path:
    sys.path.insert(0, NOTEBOOK_DIR)

from IPython.display import display, HTML, update_display

os.environ["IMO_NORMALIZE_CLIENT_ID"] = ""
os.environ["IMO_NORMALIZE_SECRET"]    = ""
os.environ["IMO_KG_CLIENT_ID"]        = ""
os.environ["IMO_KG_CLIENT_SECRET"]    = ""

os.environ["AWS_ACCESS_KEY_ID"]     = ""
os.environ["AWS_SECRET_ACCESS_KEY"] = ""
os.environ["AWS_SESSION_TOKEN"]     = ""
os.environ["AWS_DEFAULT_REGION"]    = "us-east-1"

import kg_config as kc
print("✓  Credentials loaded")

✓  Credentials loaded


## 1. LLM, Tools & System Prompt

In [2]:
from langchain_aws import ChatBedrockConverse
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage
from kg_api_client import KGApiClient
from typing import Any

llm = ChatBedrockConverse(
    model_id=os.getenv("MODEL_ID", "us.anthropic.claude-haiku-4-5-20251001-v1:0"),
    region_name=os.getenv("AWS_REGION", "us-east-1"),
    temperature=0,
    provider="anthropic",
    max_tokens=64000,
)
client = KGApiClient()

# --- Token overflow prevention (from route's _cap_tool_response) ---
_KG_LIST_THRESHOLD = 200
_MAX_TOOL_RESPONSE_CHARS = 100000

def _truncate_kg_lists(data, candidate_problems=None):
    if not isinstance(data, dict):
        return data
    list_fields = [
        "treatedProblems", "causedProblems", "domainNarrower", "narrower",
        "synonyms", "associatedTreatments", "contraindicatedTreatments",
        "supportiveTreatments", "domainNarrowerByMedications",
        "domainNarrowerByProcedures", "containingMedications",
    ]
    result = {}
    for key, value in data.items():
        if key in list_fields and isinstance(value, list) and len(value) > _KG_LIST_THRESHOLD:
            total_count = len(value)
            if candidate_problems:
                candidates_lower = [c.lower().strip() for c in candidate_problems]
                matched = [
                    item for item in value
                    if any(
                        cand in item.get("title", "").lower()
                        or item.get("title", "").lower() in cand
                        for cand in candidates_lower
                    )
                ]
                result[key] = matched
                result[key + "_summary"] = (
                    "KG returned " + str(total_count) + " total items. "
                    + "Showing " + str(len(matched)) + " items matching candidates. "
                    + "Full list too large (" + str(total_count) + " items)."
                )
            else:
                result[key] = value[:_KG_LIST_THRESHOLD]
                result[key + "_summary"] = (
                    "KG returned " + str(total_count) + " total items. "
                    + "Showing first " + str(_KG_LIST_THRESHOLD) + " of " + str(total_count) + ". "
                    + "Truncated to prevent token overflow."
                )
        elif key == "lexical" and isinstance(value, dict):
            result[key] = _truncate_kg_lists(value, candidate_problems)
        else:
            result[key] = value
    return result

def _aggressive_truncate(data, max_items):
    if not isinstance(data, dict):
        return data
    result = {}
    for key, value in data.items():
        if isinstance(value, list) and len(value) > max_items:
            total = len(value)
            result[key] = value[:max_items]
            result[key + "_summary"] = (
                "KG returned " + str(total) + " total items. "
                + "Showing first " + str(max_items) + " of " + str(total)
                + " (truncated for token limit)."
            )
        elif isinstance(value, dict):
            result[key] = _aggressive_truncate(value, max_items)
        else:
            result[key] = value
    return result

def _cap_tool_response(result, candidate_problems=None):
    result = _truncate_kg_lists(result, candidate_problems)
    output = json.dumps(result, indent=2)
    if len(output) <= _MAX_TOOL_RESPONSE_CHARS:
        return output
    shrink_limit = 50
    while len(output) > _MAX_TOOL_RESPONSE_CHARS and shrink_limit >= 10:
        result = _aggressive_truncate(result, shrink_limit)
        output = json.dumps(result, indent=2)
        shrink_limit = shrink_limit // 2
    if len(output) > _MAX_TOOL_RESPONSE_CHARS:
        result = _aggressive_truncate(result, 5)
        output = json.dumps(result, indent=2)
    return output

# --- Tool definitions (exact match with refine_with_kg route) ---

@tool
def normalize_medical_term(input_term: str, domain: str = "Medication", order_name: str = "") -> str:
    """Normalize a medical term using IMO Precision Normalize API.

    Args:
        input_term: The clinical term (e.g., "Metformin", "diabetes mellitus", "glucose"). Must not be empty.
        domain: "Medication" for drugs, "Problem" for diagnoses, "Lab" for lab results. Default is Medication.
        order_name: Optional lab order name (e.g., "lipid panel", "comprehensive metabolic panel"). Only used when domain="Lab". If you only have the order name and not the component name, pass the order name as input_term instead.
    """
    if not input_term or not input_term.strip():
        return json.dumps({"success": False, "error": "STOP: input_term is empty. Do NOT retry with empty string. Do NOT call this tool again without a valid medication name. Skip and proceed to scoring."})
    result = client.normalize_medical_term(input_term.strip(), domain, order_name=order_name.strip() if order_name else "")
    return json.dumps(result, indent=2)

@tool
def get_containing_medications(imo_lexical_code: str) -> str:
    """Step 1 of medication identification: get dose forms and root-level containing medications.

    Returns:
    - domainNarrower: flat list of dose forms/routes/strengths/brand names (code + title only)
    - containingMedications (root): fully specified medications at ingredient level (SCD/SBD)

    Single substance (no "/" in name): use root containingMedications directly.
    Multiple substance ("/" in name): pass each domainNarrower code to get_containing_medications.

    Args:
        imo_lexical_code: The default_lexical_code of the medication (from normalize with domain="Medication"). REQUIRED.
    """
    if not imo_lexical_code or not imo_lexical_code.strip():
        return json.dumps({"success": False, "error": "STOP: imo_lexical_code is empty. Normalize the medication first to get its default_lexical_code."})
    result = client.get_containing_medications(imo_lexical_code.strip())
    return _cap_tool_response(result)

@tool
def get_hierarchies_for_containing_medications(imo_lexical_codes: str) -> str:
    """Batch version of get_hierarchy_for_containing_medication.

    Pass ALL containingMedication codes at once as a comma-separated string.
    Internally calls get_hierarchy_for_containing_medication for every code and
    returns the combined results. Use this instead of looping — one call covers all.

    - Single substance: pass ALL root containingMedication codes from get_containing_medications (Step 3)
    - Multiple substance: pass ALL containingMedication codes from ALL get_containing_medications calls in Step 4a

    Returns a list of hierarchies, one entry per code, each with a lexical object
    containing 3-level domainNarrower (strengths, dose forms, SCD/SBD names).

    Args:
        imo_lexical_codes: Comma-separated containingMedication codes. REQUIRED.
                           Example: "1234567,8901234,5678901"
    """
    if not imo_lexical_codes or not imo_lexical_codes.strip():
        return json.dumps({"success": False, "error": "STOP: imo_lexical_codes is empty. Pass comma-separated containingMedication codes."})

    codes = [c.strip() for c in imo_lexical_codes.split(",") if c.strip()]
    if not codes:
        return json.dumps({"success": False, "error": "STOP: No valid codes parsed from imo_lexical_codes."})

    results = []
    errors = []
    for code_val in codes:
        result = client.get_hierarchy_for_containing_medication(code_val)
        if result.get("success"):
            results.append({"code": code_val, "lexical": result.get("lexical")})
        else:
            errors.append({"code": code_val, "error": result.get("error", "Unknown error")})

    response = {
        "success": True,
        "total": len(codes),
        "found": len(results),
        "hierarchies": results,
    }
    if errors:
        response["errors"] = errors
    return _cap_tool_response(response)

MED_ID_TOOLS = [normalize_medical_term, get_containing_medications, get_hierarchies_for_containing_medications]

SYSTEM_PROMPT = """You are a medication identification agent. Given a medication name, you use the IMO Knowledge Graph to identify its dose forms, strengths, and fully specified medication names (SCD/SBD level).

## WHAT YOU CAN ANSWER

1. **What are the dose forms of medication X?** (e.g., Oral Tablet, Oral Capsule, Injectable Solution)
2. **What are the strengths of medication X?** (e.g., 500 MG, 250 MG, 125 MG)
3. **For medication X and dose form Y, what specific meds are available?** (e.g., amoxicillin 500 MG Oral Capsule)
4. **For medication X and strength Y, what specific meds are available?** (e.g., amoxicillin 500 MG Oral Capsule, amoxicillin 500 MG Oral Tablet)

## WORKFLOW

You MUST follow every step in order. Do NOT skip any step. Do NOT assume results from a prior step — always use actual tool output.

### Step 1: Extract the Medication Name
From the user's question, extract the medication name. Note any dose form or strength filter the user asked about.

**Determine substance type — check the input name for "/":**
- **Single substance**: no "/" in name (e.g., "amoxicillin", "carvedilol")
- **Multiple substance**: "/" present in name (e.g., "amoxicillin/clavulanate")

State the substance type explicitly before proceeding.

### Step 2: Normalize the Medication
MUST call: normalize_medical_term(input_term=<medication name>, domain="Medication")
Use the returned `default_lexical_code` in Step 3. Do NOT proceed without it.

### Step 3: Get Containing Medications
MUST call: get_containing_medications(imo_lexical_code=<default_lexical_code from Step 2>)

The response contains:
- **`domainNarrower[]`**: flat list of formulation variants (code + title)
- **`containingMedications[]`** (root): medications at ingredient level

Record EVERY code from both lists. Do NOT filter or discard any item at this stage.

### Step 4: Branch by Substance Type

#### Single Substance (no "/" in input name)

From Step 3, take the **complete list of `containingMedications`** (every item, no exceptions).
Write out all codes as a comma-separated string, then make exactly **ONE** call:
get_hierarchies_for_containing_medications(imo_lexical_codes=<ALL containingMedication codes, comma-separated>)

**RULES for this call:**
- Include EVERY containingMedication code — do NOT drop, sample, or skip any
- Make exactly ONE call with all codes together — do NOT call it multiple times
- Do NOT proceed to Step 5 until this call completes

#### Multiple Substance ("/" present in input name)

**Step 4a — MUST call get_containing_medications for EVERY domainNarrower code from Step 3:**
For each `domainNarrower` code (every single one, no sampling):
  get_containing_medications(imo_lexical_code=<domainNarrower code>)
  Returns `containingMedications` for that formulation

**RULES for Step 4a:**
- Call for EVERY domainNarrower code — do NOT skip any, do NOT sample
- Complete ALL Step 4a calls before moving to Step 4b
- Accumulate ALL `containingMedication` codes across all Step 4a responses

**Step 4b — after ALL Step 4a calls are done:**
Collect every `containingMedication` code from every Step 4a response into one comma-separated string, then make exactly **ONE** call:
get_hierarchies_for_containing_medications(imo_lexical_codes=<ALL containingMedication codes, comma-separated>)

**RULES for Step 4b:**
- Include codes from ALL Step 4a results — do NOT omit any
- Make exactly ONE call — do NOT call it multiple times

### Step 5: Parse and Categorize the Results

The `get_hierarchies_for_containing_medications` response has this structure:
```
{
  "hierarchies": [
    {
      "code": "<containingMedication code>",
      "lexical": {
        "title": "<containingMedication title>",
        "domainNarrower": [ ... ]
      }
    },
    ...
  ]
}
```

**MUST process EVERY item in `hierarchies[]` — do NOT skip any.**

**Classify using KG structure only — do NOT use title string matching:**

- `lexical.domainNarrower` is **empty or null** this item is a **leaf node**
  `lexical.title` + `lexical.code` = a **Fully Specified (SCD/SBD)** medication — add to Fully Specified list

- `lexical.domainNarrower` is **non-empty** this item is a **parent node**
  `lexical.title` = a Dose Form or Strength — add to Dose Forms / Strengths list
  every item inside `lexical.domainNarrower` at any nesting level = a **Fully Specified (SCD/SBD)** medication — add each to Fully Specified list

**After processing all items, deduplicate each list by code** (same SCD may appear under multiple parents).

### Step 6: Present Results

#### Medication Identification Results

**Medication:** (normalized title) (`code`)

**Dose Forms Available:**

| # | Dose Form | Code |
|---|---|---|
| 1 | (title) | (code) |

**Strengths Available:**

| # | Strength | Code |
|---|---|---|
| 1 | (title) | (code) |

**Fully Specified Medications (SCD/SBD):**

| # | Medication Name | Code |
|---|---|---|
| 1 | (title) | (code) |

If the user asked a FILTERED question (e.g., "for amoxicillin and Oral Capsule, what meds are available?"), present ONLY the items matching that filter, with a note about what filter was applied.

## RULES
- Follow all steps in order — do NOT skip, reorder, or combine steps.
- Do NOT assume tool results — always use actual returned data.
- Do NOT sample or pick a "representative subset" — always process every item.
- Always normalize the medication first — never invent codes.
- ONLY present data returned by the KG. Never fabricate medication names or codes.
- If the KG returns empty results, state "No data found in KG for this medication" — do NOT invent results.
- If normalize fails, report the error and stop.
- Before each tool call, briefly state what you are doing and which step you are on.
- When filtering by dose form or strength, match by checking if the filter term appears in the item's title (case-insensitive).
- Deduplicate final lists by code before presenting.

## SUMMARY BLOCK (REQUIRED)
After your final answer, include a summary block wrapped in `[SUMMARY]` and `[/SUMMARY]` tags. This is displayed in a visual explainer panel for non-technical audiences. Write in simple, plain language.

**FORMAT: Exactly 4 paragraphs separated by blank lines.** Each paragraph maps to a visual card:

1. **Paragraph 1 — Clinical Question:** Restate what the user asked in 1-2 plain sentences. Mention the medication being identified.
2. **Paragraph 2 — What the Agent Did:** Describe the steps: normalized the medication, queried the Knowledge Graph for dose forms, strengths, and fully specified names (SCD/SBD).
3. **Paragraph 3 — Knowledge Graph Value:** The KG provided the complete medication hierarchy — dose forms, strengths, and fully specified names (SCD/SBD). Emphasize that these are **curated, clinically-verified relationships** that cannot be derived from text matching — this is the KG's unique value.
4. **Paragraph 4 — Recommendation:** Summarize what was identified for the medication.

Keep each paragraph concise (2-4 sentences max). Bold key terms."""

print("✓  LLM, Tools, and System Prompt ready")
print("   max_tokens=64000, _MAX_TOOL_RESPONSE_CHARS=100000")
print("   Tools: " + str([t.name for t in MED_ID_TOOLS]))

✓  LLM, Tools, and System Prompt ready
   max_tokens=64000, _MAX_TOOL_RESPONSE_CHARS=100000
   Tools: ['normalize_medical_term', 'get_containing_medications', 'get_hierarchies_for_containing_medications']


## 2. HTML Renderer + Streaming Engine

CSS ported from `chat_med_refinement.html` + Python functions that render
agent activity into rich HTML panels. Supports **live streaming** via `update_display()`.

In [3]:
def esc(text):
    return html_mod.escape(str(text)) if text else ""

AGENT_CSS = """
<style>
:root {
    --primary: #57039E; --primary-light: #7C0EDD; --primary-dark: #3D0270;
    --accent: #10b981; --danger: #ef4444;
    --bg: #f3f4f6; --bg-card: #ffffff; --text: #1e293b; --text-secondary: #64748b; --border: #e5e7eb;
}
.agent-panel { font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif; color:var(--text); line-height:1.6; margin:1rem 0; }
.agent-panel * { box-sizing:border-box; }
.msg-container { display:flex; gap:0.75rem; margin-bottom:1.25rem; animation:fadeIn 0.3s ease-in; }
@keyframes fadeIn { from{opacity:0;transform:translateY(8px)} to{opacity:1;transform:translateY(0)} }
.msg-avatar { width:36px;height:36px;border-radius:50%;display:flex;align-items:center;justify-content:center;font-size:1.2rem;flex-shrink:0;background:var(--accent); }
.msg-content { flex:1;background:var(--bg-card);border-radius:12px;padding:1rem;box-shadow:0 1px 2px rgba(0,0,0,0.05); }
.msg-header { display:flex;justify-content:space-between;align-items:center;margin-bottom:0.5rem; }
.msg-sender { font-weight:600;color:var(--text);font-size:0.9rem; }
.msg-time { font-size:0.75rem;color:var(--text-secondary); }
.msg-user .msg-avatar { background:var(--primary); }
.msg-user .msg-content { background:var(--primary);color:#fff; }
.msg-user .msg-sender,.msg-user .msg-time { color:rgba(255,255,255,0.8); }
.reasoning-steps { margin-bottom:1rem;border:1px solid var(--border);border-radius:10px;overflow:hidden;background:var(--bg); }
.reasoning-hdr { display:flex;justify-content:space-between;align-items:center;padding:0.6rem 1rem;background:#f0e7ff;cursor:pointer;user-select:none;border-bottom:1px solid var(--border); }
.reasoning-hdr h4 { font-size:0.88rem;color:var(--primary-dark);margin:0; }
.step-count { font-size:0.68rem;font-weight:600;padding:0.1rem 0.45rem;border-radius:999px;background:var(--primary);color:#fff; }
.toggle-icon { font-size:0.7rem;color:var(--text-secondary);transition:transform 0.2s; }
.reasoning-body { padding:0.6rem;display:flex;flex-direction:column;gap:0.6rem; }
.step { padding:0.7rem 0.85rem;background:var(--bg-card);border-radius:8px;position:relative; }
.step-reasoning { border-left:4px solid #f59e0b;background:linear-gradient(135deg,#fffbeb,var(--bg-card)); }
.step-tool { border-left:4px solid #6d28d9;background:linear-gradient(135deg,#ede9fe,#f5f3ff);box-shadow:0 1px 3px rgba(109,40,217,0.1); }
.step-tool.normalize { border-left-color:#0891b2;background:linear-gradient(135deg,#ecfeff,var(--bg-card));box-shadow:none; }
.step-result { border-left:4px solid var(--accent); }
.step-badge { position:absolute;top:0.5rem;right:0.6rem;color:#fff;font-size:0.63rem;font-weight:700;padding:0.1rem 0.4rem;border-radius:999px;text-transform:uppercase; }
.badge-kg { background:#7c3aed; }
.badge-normalize { background:#0891b2; }
.badge-done { background:var(--accent); }
.step-hdr { display:flex;align-items:center;gap:0.4rem;margin-bottom:0.25rem;font-weight:600;font-size:0.88rem; }
.step-icon { width:22px;height:22px;border-radius:50%;display:inline-flex;align-items:center;justify-content:center;font-size:0.75rem;flex-shrink:0; }
.icon-tool { background:#ede9fe;color:#7c3aed; }
.icon-normalize { background:#cffafe;color:#0e7490; }
.icon-result { background:#d1fae5;color:#065f46; }
.icon-reasoning { background:#fef3c7;color:#92400e; }
.step-detail { font-size:0.83rem;color:var(--text-secondary);padding-left:1.8rem; }
.step-reasoning .step-title { color:#92400e; }
.step-reasoning .reasoning-text { font-size:0.83rem;color:var(--text);line-height:1.5;padding-left:1.8rem; }
.step-tool.normalize .step-title { color:#0e7490; }
.raw-toggle { margin-top:0.5rem;padding-top:0.5rem;border-top:1px solid var(--border); }
.raw-toggle summary { font-size:0.75rem;font-weight:600;color:#7c3aed;cursor:pointer;padding:0.3rem 0.5rem;border-radius:6px; }
.raw-toggle summary:hover { background:rgba(87,3,158,0.05); }
.raw-toggle pre { margin:0.4rem 0 0;background:#f8f9fa;border:1px solid #e2e8f0;border-radius:6px;padding:0.6rem;max-height:300px;overflow-y:auto;font-size:0.72rem;white-space:pre-wrap;word-wrap:break-word; }
.final-answer { margin-top:0.5rem; }
.final-label { font-size:0.73rem;font-weight:700;text-transform:uppercase;letter-spacing:0.05em;color:var(--accent);margin-bottom:0.3rem;padding-bottom:0.3rem;border-bottom:2px solid var(--accent); }
.final-body { font-size:0.95rem;line-height:1.7; }
.final-body strong { color:var(--primary); }
.final-body code { background:#f1f5f9;padding:0.1rem 0.35rem;border-radius:4px;font-size:0.85em; }
.streaming-indicator { display:flex;align-items:center;gap:0.35rem;padding:0.75rem 1rem;color:var(--primary);font-weight:600;font-size:0.88rem;background:linear-gradient(135deg,rgba(87,3,158,0.04),rgba(124,14,221,0.06));border-radius:8px;margin-top:0.5rem; }
.dot-pulse { display:inline-block;width:8px;height:8px;border-radius:50%;background:var(--primary);animation:pulse 1.4s ease-in-out infinite; }
.dot-pulse:nth-child(2) { animation-delay:0.2s; }
.dot-pulse:nth-child(3) { animation-delay:0.4s; }
@keyframes pulse { 0%,100%{opacity:0.2;transform:scale(0.7)} 50%{opacity:1;transform:scale(1.2)} }
</style>
"""

TOOL_NAMES = {
    "normalize_medical_term": "Normalize Medical Term",
    "get_containing_medications": "GraphQL: Get Containing Medications",
    "get_hierarchies_for_containing_medications": "GraphQL: Get Medication Hierarchies",
}

def is_normalize(tool_name):
    return tool_name == "normalize_medical_term"

def fmt_tool_input(tool_name, args):
    if tool_name == "normalize_medical_term":
        return (
            'Normalizing <strong>"' + esc(args.get("input_term", ""))
            + '"</strong> in domain <strong>' + esc(args.get("domain", "Medication")) + '</strong>'
        )
    elif tool_name == "get_containing_medications":
        return (
            'Getting dose forms & containing medications for lexical code <strong>'
            + esc(args.get("imo_lexical_code", "")) + '</strong>'
        )
    elif tool_name == "get_hierarchies_for_containing_medications":
        codes = args.get("imo_lexical_codes", "")
        count = len([c for c in codes.split(",") if c.strip()]) if codes else 0
        truncated = codes[:120] + ("..." if len(codes) > 120 else "")
        return (
            'Fetching hierarchies for <strong>' + str(count)
            + '</strong> containing medication(s): <strong>' + esc(truncated) + '</strong>'
        )
    return '<pre>' + esc(json.dumps(args, indent=2)) + '</pre>'

def _make_table(headers, rows, max_height="250px", col_widths=None):
    """Build a div-based grid table that bypasses Jupyter's table CSS entirely."""
    n = len(headers)
    if not col_widths:
        col_widths = [str(round(100 / n)) + "%" for _ in range(n)]
    grid_cols = " ".join(col_widths)

    t = '<div style="overflow-x:auto;margin-top:0.3rem;max-height:' + max_height + ';overflow-y:auto;border:1px solid #e5e7eb;border-radius:6px;">'
    t += '<div style="display:grid;grid-template-columns:' + grid_cols + ';background:#57039E;">'
    for h in headers:
        t += '<div style="color:#fff;font-weight:600;text-align:left;padding:0.4rem 0.6rem;white-space:nowrap;font-size:0.8rem;">' + h + '</div>'
    t += '</div>'
    for idx, row in enumerate(rows):
        bg = "#f8fafc" if idx % 2 == 1 else "#ffffff"
        t += '<div style="display:grid;grid-template-columns:' + grid_cols + ';border-bottom:1px solid #e5e7eb;background:' + bg + ';">'
        for cell in row:
            t += '<div style="padding:0.35rem 0.6rem;color:#1e293b;font-size:0.8rem;overflow:hidden;text-overflow:ellipsis;">' + cell + '</div>'
        t += '</div>'
    t += '</div>'
    return t

def fmt_tool_result_html(tool_name, result_str):
    try:
        data = json.loads(result_str) if isinstance(result_str, str) else result_str
    except (json.JSONDecodeError, TypeError):
        return '<pre>' + esc(str(result_str)[:500]) + '</pre>'

    html_out = ""
    if tool_name == "normalize_medical_term" and isinstance(data, dict) and data.get("results"):
        matches = data["results"][0].get("matches", []) if data["results"] else []
        term = data["results"][0].get("input_term", "") if data["results"] else ""
        html_out += '<div style="font-size:0.83rem;color:var(--text-secondary);margin-bottom:0.3rem;">'
        html_out += str(len(matches)) + ' result(s) for "<strong>' + esc(term) + '</strong>"</div>'
        rows = []
        for m in matches[:5]:
            code_val = m.get("default_lexical_code", m.get("lexical_code", ""))
            icd = ", ".join(m.get("icd10_codes", []))
            rows.append([
                '<strong>' + esc(m.get("title", "")) + '</strong>',
                esc(code_val),
                "{:.3f}".format(m.get("score", 0)),
                esc(icd),
            ])
        html_out += _make_table(["IMO Description", "Lexical Code", "Score", "ICD-10"], rows,
                                col_widths=["40%", "25%", "15%", "20%"])

    elif tool_name == "get_containing_medications" and isinstance(data, dict) and data.get("lexical"):
        lex = data["lexical"]
        html_out += '<div style="font-size:0.83rem;color:var(--text-secondary);">'
        html_out += 'Medication: <strong>' + esc(lex.get("title", "")) + '</strong>'
        html_out += ' (code: ' + esc(lex.get("code", "")) + ')</div>'
        narrower = lex.get("domainNarrower", []) or []
        containing = lex.get("containingMedications", []) or []
        if narrower:
            html_out += '<div style="margin-top:0.4rem;"><strong>Domain Narrower ('
            html_out += str(len(narrower)) + '):</strong></div>'
            rows = [[esc(n.get("title", "")), esc(n.get("code", ""))] for n in narrower]
            html_out += _make_table(["Title", "Code"], rows, "200px", col_widths=["65%", "35%"])
        if containing:
            html_out += '<div style="margin-top:0.4rem;"><strong>Containing Medications ('
            html_out += str(len(containing)) + '):</strong></div>'
            rows = [[esc(c.get("title", "")), esc(c.get("code", ""))] for c in containing]
            html_out += _make_table(["Title", "Code"], rows, "200px", col_widths=["65%", "35%"])

    elif tool_name == "get_hierarchies_for_containing_medications" and isinstance(data, dict) and data.get("hierarchies"):
        hierarchies = data["hierarchies"]
        html_out += '<div style="font-size:0.83rem;color:var(--text-secondary);margin-bottom:0.3rem;">'
        html_out += '<strong>' + str(len(hierarchies)) + '</strong> hierarchy/hierarchies returned</div>'
        for h in hierarchies:
            lex = h.get("lexical") or {}
            title = lex.get("title", "")
            hcode = lex.get("code", h.get("code", ""))
            narrower = lex.get("domainNarrower", []) or []
            leaf_label = ' <span style="font-size:0.7rem;color:#065f46;background:#d1fae5;padding:0.1rem 0.3rem;border-radius:999px;">leaf</span>' if not narrower else ""
            html_out += '<div style="margin:0.3rem 0 0.1rem;padding:0.3rem 0.5rem;background:#f8fafc;'
            html_out += 'border:1px solid #e5e7eb;border-radius:6px;">'
            html_out += '<strong>' + esc(title) + '</strong> '
            html_out += '<span style="font-size:0.7rem;background:#57039E;color:#fff;'
            html_out += 'padding:0.1rem 0.3rem;border-radius:999px;">' + esc(hcode) + '</span>'
            html_out += leaf_label + '</div>'
            if narrower:
                rows = []
                for n in narrower:
                    children = n.get("domainNarrower", []) or []
                    child_count = str(len(children)) if children else "0"
                    rows.append([esc(n.get("title", "")), esc(n.get("code", "")), child_count])
                html_out += _make_table(["Title", "Code", "Children"], rows, "200px",
                                        col_widths=["55%", "30%", "15%"])
        errors = data.get("errors", [])
        if errors:
            html_out += '<div style="margin-top:0.3rem;color:#ef4444;font-size:0.8rem;">'
            html_out += '<strong>' + str(len(errors)) + ' error(s):</strong> '
            html_out += esc(str(errors[:3])) + '</div>'
    else:
        html_out += '<pre style="max-height:200px;overflow:auto;font-size:0.72rem;">'
        html_out += esc(json.dumps(data, indent=2)[:1000]) + '</pre>'

    raw_json = json.dumps(data, indent=2)
    html_out += '<details class="raw-toggle"><summary>View Response JSON</summary>'
    html_out += '<pre>' + esc(raw_json[:3000]) + '</pre></details>'
    return html_out

def fmt_markdown_to_html(text):
    text = re.sub(r'\[SUMMARY\][\s\S]*?\[/SUMMARY\]', '', text).strip()
    lines = text.split('\n')
    result = []
    in_table = False
    table_rows = []
    for line in lines:
        stripped = line.strip()
        if stripped.startswith('|') and stripped.endswith('|'):
            if re.match(r'^\|[\s\-:|]+\|$', stripped):
                continue
            if not in_table:
                in_table = True
                table_rows = []
            table_rows.append([c.strip() for c in stripped.split('|')[1:-1]])
        else:
            if in_table:
                result.append(_build_table(table_rows))
                in_table = False
            result.append(stripped)
    if in_table:
        result.append(_build_table(table_rows))
    f = '\n'.join(result)
    f = re.sub(r'^#### (.+)$', r'<h4>\1</h4>', f, flags=re.M)
    f = re.sub(r'^### (.+)$', r'<h4>\1</h4>', f, flags=re.M)
    f = re.sub(r'^## (.+)$', r'<h3>\1</h3>', f, flags=re.M)
    f = re.sub(r'\*\*(.+?)\*\*', r'<strong>\1</strong>', f)
    f = re.sub(r'\*(.+?)\*', r'<em>\1</em>', f)
    f = re.sub(r'`(.+?)`', r'<code>\1</code>', f)
    f = f.replace('\n\n', '</p><p>').replace('\n', '<br>')
    return '<p>' + f + '</p>'

def _build_table(rows):
    if not rows:
        return ''
    headers = rows[0]
    data_rows = rows[1:]
    n = len(headers)
    if n == 2:
        widths = ["65%", "35%"]
    elif n == 3:
        widths = ["50%", "30%", "20%"]
    elif n == 4:
        widths = ["40%", "25%", "15%", "20%"]
    else:
        widths = None
    return _make_table(headers, data_rows, "300px", col_widths=widths)

def _streaming_dots(msg="Agent is working..."):
    return (
        '<div class="streaming-indicator">'
        '<span class="dot-pulse"></span><span class="dot-pulse"></span><span class="dot-pulse"></span>'
        '<span style="margin-left:0.3rem;">' + esc(msg) + '</span>'
        '</div>'
    )

def render_agent_run(activity, final_text, label="Medication Identification Results", streaming=False):
    normalize_n = sum(1 for a in activity if a["type"] == "tool_call" and is_normalize(a["tool"]))
    kg_n = sum(1 for a in activity if a["type"] == "tool_call" and not is_normalize(a["tool"]))
    count_parts = []
    if normalize_n:
        count_parts.append(str(normalize_n) + " Normalize")
    if kg_n:
        count_parts.append(str(kg_n) + " KG")
    count_label = " · ".join(count_parts) if count_parts else "0 calls"

    norm_idx = 0
    kg_idx = 0
    steps_html = ""
    for a in activity:
        if a["type"] == "reasoning" and a.get("text", "").strip():
            steps_html += (
                '<div class="step step-reasoning">'
                '<div class="step-hdr"><span class="step-icon icon-reasoning">\U0001f9e0</span>'
                '<span class="step-title">Agent Reasoning</span></div>'
                '<div class="reasoning-text">' + fmt_markdown_to_html(a["text"]) + '</div>'
                '</div>'
            )
        elif a["type"] == "tool_call":
            tool_name = a["tool"]
            norm = is_normalize(tool_name)
            if norm:
                norm_idx += 1
                idx = norm_idx
            else:
                kg_idx += 1
                idx = kg_idx
            cls = "step-tool normalize" if norm else "step-tool"
            badge_cls = "badge-normalize" if norm else "badge-kg"
            badge_lbl = "Normalize" if norm else "KG Call"
            icon_cls = "icon-normalize" if norm else "icon-tool"
            icon = "⚙" if norm else "\U0001f50d"
            display_name = TOOL_NAMES.get(tool_name, tool_name)
            detail = fmt_tool_input(tool_name, a.get("args", {}))
            steps_html += (
                '<div class="step ' + cls + '">'
                '<span class="step-badge ' + badge_cls + '">' + badge_lbl + ' ' + str(idx) + '</span>'
                '<div class="step-hdr"><span class="step-icon ' + icon_cls + '">' + icon + '</span>'
                '<span class="step-title">' + esc(display_name) + '</span>'
                '<span class="step-badge badge-done" style="position:static;margin-left:auto;">Done</span></div>'
                '<div class="step-detail">' + detail + '</div>'
                '</div>'
            )
        elif a["type"] == "tool_result":
            tool_name = a["tool"]
            display_name = TOOL_NAMES.get(tool_name, tool_name)
            result_html = fmt_tool_result_html(tool_name, a.get("result", ""))
            steps_html += (
                '<div class="step step-result">'
                '<div class="step-hdr"><span class="step-icon icon-result">☰</span>'
                '<span class="step-title">Results from ' + esc(display_name) + '</span></div>'
                '<div>' + result_html + '</div>'
                '</div>'
            )

    if final_text:
        bottom = (
            '<div class="final-answer">'
            '<div class="final-label">' + esc(label) + '</div>'
            '<div class="final-body">' + fmt_markdown_to_html(final_text) + '</div>'
            '</div>'
        )
    elif streaming:
        bottom = _streaming_dots("Agent is working...")
    else:
        bottom = ""

    if not activity and streaming:
        reasoning_section = _streaming_dots("Agent is thinking...")
    else:
        reasoning_section = (
            '<div class="reasoning-steps">'
            '<div class="reasoning-hdr" onclick="var b=this.parentElement.querySelector('
            "'.reasoning-body');b.style.display=b.style.display==='none'?'flex':'none'" + '">'
            '<h4>Agent Reasoning &amp; API Calls</h4>'
            '<span class="step-count">' + count_label + '</span>'
            '<span class="toggle-icon">▼</span>'
            '</div>'
            '<div class="reasoning-body">' + steps_html + '</div>'
            '</div>'
        )

    return (
        AGENT_CSS
        + '<div class="agent-panel">'
        '<div class="msg-container">'
        '<div class="msg-avatar">\U0001f916</div>'
        '<div class="msg-content">'
        '<div class="msg-header">'
        '<span class="msg-sender">Medication Identification Agent</span>'
        '</div>'
        + reasoning_section
        + bottom
        + '</div></div></div>'
    )

print("✓  HTML renderer + streaming engine ready")

✓  HTML renderer + streaming engine ready


## 3. Agent + Streaming Runner

Creates the ReAct agent and provides `stream_agent()` — streams via
`agent.stream(stream_mode='values')` and live-updates the HTML display
after every agent/tool step.

In [4]:
from langgraph.prebuilt import create_react_agent

agent = create_react_agent(llm, tools=MED_ID_TOOLS, prompt=SYSTEM_PROMPT)

_conversations = {}
_NOTE_MIN_WORDS = 30

def _extract_text(content):
    """Extract text from msg.content (str or list of content blocks)."""
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for block in content:
            if isinstance(block, str):
                parts.append(block)
            elif isinstance(block, dict) and block.get("type") == "text":
                parts.append(block.get("text", ""))
        return "\n".join(parts)
    return str(content) if content else ""

def _build_activity(messages, skip_count=0):
    """Parse messages into (activity_list, final_text). Skip first skip_count messages."""
    activity = []
    final_text = ""
    for msg in messages[skip_count:]:
        role = msg.type if hasattr(msg, 'type') else 'unknown'
        if role == 'ai':
            text = _extract_text(msg.content)
            if text.strip():
                if hasattr(msg, 'tool_calls') and msg.tool_calls:
                    activity.append({"type": "reasoning", "text": text})
                else:
                    final_text = text
            if hasattr(msg, 'tool_calls') and msg.tool_calls:
                for tc in msg.tool_calls:
                    activity.append({"type": "tool_call", "tool": tc["name"], "args": tc.get("args", {})})
        elif role == 'tool':
            content = msg.content if isinstance(msg.content, str) else json.dumps(msg.content)
            activity.append({"type": "tool_result", "tool": msg.name, "result": content})
    return activity, final_text

def stream_agent(user_message, session_id="notebook", label="Medication Identification Results"):
    """Stream agent execution with live HTML updates, conversation history, and error handling."""
    mode = "med-id"
    session_key = session_id + "_" + mode
    is_new_clinical_note = len(user_message.split()) >= _NOTE_MIN_WORDS
    if is_new_clinical_note and session_key in _conversations and _conversations[session_key]:
        _conversations[session_key] = []

    history = _conversations.get(session_key, [])
    messages = list(history) + [HumanMessage(content=user_message)]

    input_messages = [SystemMessage(content=SYSTEM_PROMPT)] + messages
    skip_count = len(input_messages)

    display_id = "medid_" + uuid.uuid4().hex[:6]
    display(HTML(render_agent_run([], "", label=label, streaming=True)), display_id=display_id)

    final_text = ""
    try:
        last_state = None
        for state in agent.stream(
            {"messages": input_messages},
            stream_mode="values",
            config={"recursion_limit": 100},
        ):
            last_state = state
            activity, final_text = _build_activity(state["messages"], skip_count=skip_count)
            html = render_agent_run(activity, final_text, label=label, streaming=not bool(final_text))
            update_display(HTML(html), display_id=display_id)

        all_messages = last_state["messages"] if last_state else input_messages
        activity, final_text = _build_activity(all_messages, skip_count=skip_count)
        update_display(HTML(render_agent_run(activity, final_text, label=label, streaming=False)), display_id=display_id)

        _conversations.setdefault(session_key, [])
        _conversations[session_key].append(HumanMessage(content=user_message))
        if final_text:
            from langchain_core.messages import AIMessage
            _conversations[session_key].append(AIMessage(content=final_text))
        if len(_conversations[session_key]) > 4:
            _conversations[session_key] = _conversations[session_key][-2:]

    except Exception as exc:
        error_html = (
            '<div style="background:#fce8e6;border:1px solid #c5221f;border-radius:8px;'
            'padding:12px 16px;margin:8px 16px;font-size:0.85rem;">'
            '<strong style="color:#c5221f;">Error:</strong> ' + esc(str(exc)) + '</div>'
        )
        update_display(HTML(AGENT_CSS + error_html), display_id=display_id)
        raise

    return final_text

print("✓  Agent + streaming runner ready (conversation history + error handling)")
print("   recursion_limit=100, _NOTE_MIN_WORDS=30")

✓  Agent + streaming runner ready (conversation history + error handling)
   recursion_limit=100, _NOTE_MIN_WORDS=30


D:\Users\nthonte\AppData\Local\Temp\1\ipykernel_6468\145874590.py:3: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools=MED_ID_TOOLS, prompt=SYSTEM_PROMPT)


## 4. Medication Query

In [5]:
MEDICATION_QUERY = "What are all the dose forms, strengths, and fully specified medications available for amoxicillin?"

user_html = (
    AGENT_CSS
    + '<div class="agent-panel"><div class="msg-container msg-user">'
    + '<div class="msg-avatar">\U0001f464</div>'
    + '<div class="msg-content">'
    + '<div class="msg-header"><span class="msg-sender">You</span></div>'
    + '<div style="font-size:1.05rem;">' + esc(MEDICATION_QUERY) + '</div>'
    + '</div></div></div>'
)
display(HTML(user_html))

## 5. Run Agent — Medication Identification (Streaming)

Sends the medication query and streams the full agent workflow:
1. **Normalize** the medication name via IMO Precision Normalize API
2. **Get containing medications** from the Knowledge Graph
3. **Get hierarchies** for all containing medications (dose forms, strengths, SCD/SBD)
4. **Present results** — dose forms, strengths, and fully specified medications

In [6]:
result = stream_agent(MEDICATION_QUERY)